# C-DOT UPF Closed-Loop Lab

**90-minute control-room workshop · 28th · 11:30–13:00**

Synthetic, deterministic simulation—not a live C-DOT network. The goal is one defensible control cycle, not a production claim.

In [ ]:
from pathlib import Path
import json, sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import HTML, display
from workshop import lab

display(HTML('''<style>
:root { --lab-ink:#17343f; --lab-teal:#087f8c; --lab-violet:#753bbd; --lab-amber:#b77b10; }
.jp-Notebook { max-width: 1120px; margin:auto; }
.lab-banner { border-left:6px solid var(--lab-teal); padding:18px 22px; background:#eef7f7; color:var(--lab-ink); }
.lab-stage { font:600 12px/1.2 "IBM Plex Mono",monospace; letter-spacing:.12em; color:var(--lab-teal); }
.lab-check { padding:12px 15px; border:1px solid #c8d6da; background:#f7fafb; color:var(--lab-ink); }
.lab-safe { border-left:5px solid #14835b; } .lab-warn { border-left:5px solid #d2931d; }
</style><div class="lab-banner"><div class="lab-stage">SYNTHETIC · DETERMINISTIC · PARTICIPANT WORKSPACE</div>
<h2>Observe → predict → certify → steer future sessions → measure</h2>
<p>This notebook creates a recommendation record. It has no presenter credentials and cannot publish to the live dashboard.</p></div>'''))


## Table roles (optional)

Pick any four: **traffic engineer**, **forecasting engineer**, **policy/safety engineer**, and **operator/reporter**. Roles guide the conversation; nothing is scored.

<div class='lab-stage'>01 / TRAFFIC</div>

## Create the event

Choose one controllable group and a surge from ×1.25 to ×8. Offered demand is generated independently of what the UPFs can carry.

In [ ]:
# TODO 1 — choose a traffic group and surge multiplier, then run this cell.
selected_group = "stadium|social-live|1-010204"
surge_multiplier = 4.0

event = lab.create_traffic_event(selected_group, surge_multiplier)
traffic = lab.simulate_event(event)
display(HTML(lab.traffic_plot(traffic)))
print(f"VALID EVENT · {event.group_label} · ×{event.surge_multiplier:.1f}")


**Hint 1** · Keep the stadium group for the canonical story, or run `lab.group_options()` to see all stable group IDs. If offered and carried separate during the surge, the difference is overload/loss—not hidden demand.

In [ ]:
# Collapsed solution 1
# selected_group = 'stadium|social-live|1-010204'
# surge_multiplier = 4.0

<div class='lab-stage'>02 / SIMULATE</div>

## Read the traffic correctly

- **Offered demand**: what sessions attempted to send.
- **Carried traffic**: what the network actually transported.
- **Overload**: offered load above the available service envelope.
- **Loss**: offered load not carried after queue/admission effects.

<div class='lab-check lab-warn'><strong>Checkpoint:</strong> carried throughput is an outcome. It is not a clean demand-training label when the network is constrained.</div>

<div class='lab-stage'>03 / FORECAST</div>

## Forecast before the event

Issue a causal six-window moving average. The feature window must end at or before the target starts. Compare the median with a conservative p90 planning choice.

In [ ]:
# TODO 2 — choose p50 or p90. The forecast always uses the six windows before the target.
planning_risk = "p90"

forecast = lab.causal_ma_forecast(traffic, event, planning_risk=planning_risk, history_windows=6)
planned = getattr(forecast.new_load_ul_mbps, planning_risk)
actual = traffic[event.start_window]["offered_ul_mbps"]
print(f"CAUSAL FORECAST · source ends {forecast.source_window_end.isoformat()}")
print(f"p50={forecast.new_load_ul_mbps.p50:.1f} · p90={forecast.new_load_ul_mbps.p90:.1f} · actual={actual:.1f} UL Mbps")
print(f"PLAN ON {planning_risk.upper()} · {planned:.1f} UL Mbps")


**Hint 2** · Use `planning_risk = 'p50'` for the central estimate or `'p90'` for a conservative load projection. Neither quantile guarantees that an unannounced flash crowd will be covered.

In [ ]:
# Collapsed solution 2
# planning_risk = 'p90'
# forecast = lab.causal_ma_forecast(traffic, event, planning_risk=planning_risk, history_windows=6)

<div class='lab-stage'>04 / CERTIFY + OPTIMIZE</div>

## Make a safe recommendation

The independent gate checks causality, health, group eligibility, locality, finite `[0,1]` weights, normalization, directional/session capacity, and new-session-only scope.

In [ ]:
# TODO 3 — choose static, reactive, or cohort-mpc; optionally edit the normalized weights.
controller = "cohort-mpc"
weights = lab.recommended_weights(controller, event, planning_risk)

certification = lab.certify_recommendation(
    forecast, event, controller=controller, planning_risk=planning_risk, weights=weights
)
print(certification.message)
print("requested:", certification.requested_weights)
print("applied:  ", certification.applied_weights)
print("existing sessions anchored:", certification.existing_sessions_anchored)


**Hint 3** · Start with `lab.recommended_weights(controller, event, planning_risk)`. Try editing one destination or setting `migrate_existing=True`; any unsafe recommendation must retain the last safe/static policy.

In [ ]:
# Collapsed solution 3
# controller = 'cohort-mpc'
# weights = lab.recommended_weights(controller, event, planning_risk)

### Required safety drill

Run the invalid recommendation below once. The visible rejection is a feature: the live presenter should show at least one team fallback.

In [ ]:
# Safety drill — deliberately invalid: UPF-Z is unknown and weights do not normalize.
unsafe = lab.certify_recommendation(
    forecast,
    event,
    controller=controller,
    planning_risk=planning_risk,
    weights={"upf-a": 0.55, "upf-z": 0.55},
)
print(unsafe.message)
print("fallback applied:", unsafe.applied_weights)


<div class='lab-stage'>05 / EVALUATE</div>

## Close the modeled loop

Measure the later consequence without rewriting earlier telemetry, then emit the small `WorkshopDecision` handoff. The presenter—not this notebook—translates one team recommendation into dashboard controls.

In [ ]:
outcome = lab.close_loop(traffic, event, certification)
explanation = (
    f"We chose {controller} with {planning_risk} planning because uncertainty and residual capacity "
    "matter; only newly arriving sessions may follow the accepted weights."
)
decision = lab.build_decision(
    event, certification, outcome,
    controller=controller, planning_risk=planning_risk, explanation=explanation,
)
decision_path = lab.save_decision(decision)
print(json.dumps(outcome, indent=2))
print(f"\nDECISION SAVED · {decision_path}")


<div class='lab-check lab-safe'><strong>Say it precisely:</strong> the selected policy changed placement for future sessions and reduced modeled exposure in this synthetic trace. Established sessions were not migrated. This is not guaranteed overload prevention or production readiness.</div>

## Table close

Complete one sentence: **“We would deploy this in advisory mode only after ___.”**